In [4]:
import pandas as pd

df = pd.read_csv('../data/raw/complaints.csv', low_memory=False)
print(df.shape)
df.head(3)


(2023066, 11)


,Unnamed: 0,product_5,narrative,Product,Date received,Sub-product,Issue,Sub-issue,Company,State,Timely response?
0,234,Credit Reporting,Dear Possible Financial Inc you guyss aree rep...,Credit reporting or other personal consumer re...,2024-07-27,Credit reporting,Incorrect information on your report,Account information incorrect,Possible Financial Inc,MI,Yes
1,240,Debt Collection,"XXXX XXXX XXXX ( debt collector ), sent my boy...",Debt collection,2024-07-27,I do not know,Threatened to contact someone or share informa...,Talked to a third-party about your debt,BlueChip Financial,TX,Yes
2,257,Credit Reporting,I been receiving alerts my information was fou...,Credit reporting or other personal consumer re...,2024-07-23,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,FC HoldCo LLC,SC,Yes


In [5]:
print("Column names:")
print(df.columns.tolist())

print ("\nMissing values per column:")
print(df.isnull.sum())

Column names:
['Unnamed: 0', 'product_5', 'narrative', 'Product', 'Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State', 'Timely response?']

Missing values per column:


AttributeError: 'function' object has no attribute 'sum'

In [6]:
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
Unnamed: 0               0
product_5                0
narrative                0
Product                  0
Date received            0
Sub-product          52206
Issue                    0
Sub-issue           230559
Company                  0
State                 7344
Timely response?         0
dtype: int64


In [7]:
print("Product categories and their counts:")
print(df['Product'].value_counts())

Product categories and their counts:
Product
Credit reporting, credit repair services, or other personal consumer reports    807291
Credit reporting or other personal consumer reports                             366397
Debt collection                                                                 266842
Mortgage                                                                        119116
Credit card or prepaid card                                                     108669
Checking or savings account                                                     100447
Credit card                                                                      50372
Student loan                                                                     44241
Money transfer, virtual currency, or money service                               41503
Vehicle loan or lease                                                            32077
Credit reporting                                                                 3158

In [8]:
print("Sample complaint narratives:\n")
for i, row in df[['narrative', 'Product']].sample(5, random_state=42).iterrows():
    print(f"PRODUCT: {row['Product']}")
    print(f"TEXT: {row['narrative'][:300]}")
    print("-" * 60)

Sample complaint narratives:

PRODUCT: Credit reporting, credit repair services, or other personal consumer reports
TEXT: I have asked XXXX XXXX and experian to have XXXX XXXX XXXX, XXXX XXXX XXXX and XXXX XXXX XXXX to provide proof of contract with my signature on it stating I owe these collection companys money and if proof of contract is not provided to have them deleted immediately
------------------------------------------------------------
PRODUCT: Credit reporting, credit repair services, or other personal consumer reports
TEXT: This is my fourth endeavor to tell you that I am a victim of identity theft and I complain to question specific records in my document coming about because of the wrongdoing. The records I am questioning connect with no exchanges acquiring any possession of goods, services, or money that I have made
------------------------------------------------------------
PRODUCT: Payday loan, title loan, or personal loan
TEXT: I took out a small loan which I thought

In [9]:
# Keep only the two columns we need
df_clean = df[['narrative', 'Product']].copy()

# Rename for consistency
df_clean.columns = ['text', 'product']

# Drop any nulls just to be safe
df_clean = df_clean.dropna()

# Check final shape
print(f"Clean dataset shape: {df_clean.shape}")
print(f"\nCategory distribution:")
print(df_clean['product'].value_counts())

# Save to processed folder
df_clean.to_csv('../data/processed/complaints_clean.csv', index=False)
print("\nSaved to data/processed/complaints_clean.csv")

Clean dataset shape: (2023066, 2)

Category distribution:
product
Credit reporting, credit repair services, or other personal consumer reports    807291
Credit reporting or other personal consumer reports                             366397
Debt collection                                                                 266842
Mortgage                                                                        119116
Credit card or prepaid card                                                     108669
Checking or savings account                                                     100447
Credit card                                                                      50372
Student loan                                                                     44241
Money transfer, virtual currency, or money service                               41503
Vehicle loan or lease                                                            32077
Credit reporting                                                

In [10]:
CFPB_TO_RBI = {
    "Credit card": "Credit Cards",
    "Credit card or prepaid card": "Credit Cards",
    "Checking or savings account": "Deposit Accounts",
    "Bank account or service": "Deposit Accounts",
    "Mortgage": "Loans and Advances",
    "Student loan": "Loans and Advances",
    "Vehicle loan or lease": "Loans and Advances",
    "Consumer Loan": "Loans and Advances",
    "Payday loan": "Loans and Advances",
    "Payday loan, title loan, or personal loan": "Loans and Advances",
    "Payday loan, title loan, personal loan, or advance loan": "Loans and Advances",
    "Debt collection": "Recovery Agents",
    "Debt or credit management": "Recovery Agents",
    "Money transfer, virtual currency, or money service": "Remittance / Money Transfer",
    "Money transfers": "Remittance / Money Transfer",
    "Prepaid card": "ATM / Debit Cards",
    "Virtual currency": "Electronic Banking / Mobile",
    "Other financial service": "Others",
    "Credit reporting": "Others",
    "Credit reporting, credit repair services, or other personal consumer reports": "Others",
    "Credit reporting or other personal consumer reports": "Others",
}

# Apply mapping
df_clean['rbi_category'] = df_clean['product'].map(CFPB_TO_RBI)

# Check for any unmapped categories
print("Unmapped categories:")
print(df_clean[df_clean['rbi_category'].isna()]['product'].value_counts())

print("\nRBI category distribution:")
print(df_clean['rbi_category'].value_counts())

Unmapped categories:
Series([], Name: count, dtype: int64)

RBI category distribution:
rbi_category
Others                         1205567
Recovery Agents                 267746
Loans and Advances              227695
Credit Cards                    159041
Deposit Accounts                115332
Remittance / Money Transfer      43000
ATM / Debit Cards                 4669
Electronic Banking / Mobile         16
Name: count, dtype: int64


In [11]:
# Save the final mapped dataset
df_clean[['text', 'rbi_category']].to_csv('../data/processed/complaints_mapped.csv', index=False)

print("Saved to data/processed/complaints_mapped.csv")
print(f"Total rows: {len(df_clean):,}")
print(f"Columns: text, rbi_category")
print(f"Ready for DistilBERT training!")

Saved to data/processed/complaints_mapped.csv
Total rows: 2,023,066
Columns: text, rbi_category
Ready for DistilBERT training!
